# THPT 2026 Score Distribution Analysis

National score distribution analysis for Vietnam's 2026 High School Graduation Exam (GDPT 2018 curriculum).

**Exam structure:**
- **Compulsory:** Math (`TOAN`), Literature (`VAN`)
- **Electives:** Foreign Language (`T_ANH`), History (`SU`), Geography (`DIA`), Physics (`LI`), Chemistry (`HOA`), Biology (`SINH`), Economic & Legal Education (`KTPL`), Computer Science, Technology


## 1. Load Data

Load packages and import score dataset.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath("."))

from thpt2026_analyst import (
    load_thpt_2026_data,
    clean_thpt_2026_data,
    plot_score_distribution,
    plot_subject_grid_distributions,
    plot_correlation_matrix_grid,
    calculate_subject_statistics,
    calculate_admission_blocks,
    calculate_province_statistics,
    calculate_subject_correlations,
    detect_score_outliers,
    find_valedictorians,
    create_national_dashboard,
)


In [ ]:
raw_df, metadata = load_thpt_2026_data(local_path="data/raw_diem_thpt_2026.csv")

print(f"Total raw records: {len(raw_df):,}")
if not raw_df.empty:
    display(raw_df.head(10))


## 2. Clean & Audit Data

Standardize subject column names, map candidate SBD prefixes to 63 provinces, and audit coverage.


In [ ]:
clean_df, quality_report = clean_thpt_2026_data(raw_df)

print(f"Cleaned records: {quality_report['cleaned_records']:,}")
print(f"Official MOET baseline: {quality_report['official_candidates']:,}")
print(f"Coverage: {quality_report['coverage_pct']:.2f}% ({quality_report['coverage_label']})")

if quality_report.get("missing_rates"):
    missing_df = pd.DataFrame.from_dict(quality_report["missing_rates"], orient="index")
    display(missing_df)


## 3. Score Distribution of Subjects

Multi-panel 3x3 histogram grid comparing score distributions across subjects.


In [ ]:
grid_fig = plot_subject_grid_distributions(clean_df, save_path="output/charts/subject_distributions_grid.png")
plt.show()


## 4. Detailed Subject Score Distributions

Horizontal score distribution bar charts (0.25 point step precision).


In [ ]:
subjects = ["toan", "ngu_van", "ngoai_ngu", "vat_li", "hoa_hoc", "lich_su"]
os.makedirs("output/charts", exist_ok=True)

for subj in subjects:
    if subj in clean_df.columns and not clean_df[subj].dropna().empty:
        fig = plot_score_distribution(
            df=clean_df,
            subject=subj,
            color="#C56A0A",
            save_path=f"output/charts/dist_{subj}.png"
        )
        plt.show()


## 5. Valedictorian Analysis (Thủ Khoa)

Identification of top candidates (Thủ khoa) for admission blocks (A00, A01, B00, C00, D01) and overall national score sum.


In [ ]:
valedictorians_df = find_valedictorians(clean_df)
print("=== THPT 2026 VALEDICTORIANS (THỦ KHOA) ===")
display(valedictorians_df)


## 6. University Admission Combination Totals

Total 3-subject combination scores for university admissions:
- **A00:** Math + Physics + Chemistry (`TOAN + LI + HOA`)
- **A01:** Math + Physics + English (`TOAN + LI + T_ANH`)
- **B00:** Math + Chemistry + Biology (`TOAN + HOA + SINH`)
- **C00:** Literature + History + Geography (`VAN + SU + DIA`)
- **D01:** Math + Literature + English (`TOAN + VAN + T_ANH`)


In [ ]:
block_scores_df, block_summary = calculate_admission_blocks(clean_df)
display(block_summary)


## 7. Correlation Matrix of Test Results

Pearson correlation matrix and key takeaways across subject scores.


In [ ]:
corr_fig = plot_correlation_matrix_grid(clean_df, save_path="output/charts/correlation_matrix_grid.png")
plt.show()


### Takeaway

- **STEM & Humanities Clusters:** Strong subject ties exist among STEM subjects (`TOAN`, `LI`, `HOA`, `SINH`), and among Humanities subjects (`SU`, `DIA`).
- **Literature Independence:** Students' performance in `VAN` (Literature) is independent of STEM subjects (low correlation < 0.35).
- **Language Independence:** Performance in Language subjects (`T_ANH` - English, `VAN` - Literature) does not imply performance in STEM or Humanities.


## 8. National Subject Statistics

Summary stats for all subjects (mean, median, std, min, max, percentiles, pass & high score percentages).


In [ ]:
stats_df = calculate_subject_statistics(clean_df)
display(stats_df)

os.makedirs("output/reports", exist_ok=True)
stats_df.to_csv("output/reports/national_subject_statistics_2026.csv", index=False, encoding="utf-8-sig")


## 9. Provincial Performance

Average score metrics grouped by candidate province.


In [ ]:
prov_math_df = calculate_province_statistics(clean_df, subject="toan")

if not prov_math_df.empty:
    display(prov_math_df.head(10))
    fig_prov = plot_province_heatmap(prov_math_df, subject="toan", save_path="output/charts/province_math_heatmap.png")
    plt.show()


## 10. Score Outlier Detection

IQR and Z-score distribution assessment.


In [ ]:
math_outliers = detect_score_outliers(clean_df, subject="toan")
for k, v in math_outliers.items():
    print(f"{k}: {v}")


## 11. National Score Dashboard

Multi-panel dashboard summary.


In [ ]:
dash_fig = create_national_dashboard(
    df=clean_df,
    quality_report=quality_report,
    stats_df=stats_df,
    save_path="output/charts/national_score_dashboard_2026.png"
)
plt.show()
